In [2]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
# Output directory will be the LCBP-interannual-EMMAs grab sample data dir
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"
output_dir = repo_dir / "Data/GrabSample_data"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
importlib.reload(ep)
importlib.reload(em)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Load all RI datasets 
df_23 = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")
df_20 = pd.read_csv(data_dir / "RI18-20-IC-ICP.csv")
df_22 = pd.read_csv(data_dir / "RI22-IC-ICP-isotope.csv")
df_24 = pd.read_csv(data_dir / "RI24-IC-ICP-isotope-joined.csv")
df_25 = pd.read_csv(data_dir / "RI25-IC-ICP-isotope-joined.csv")

# Standardize core column names before combining
for df_temp in [df_20, df_22, df_23, df_24, df_25]:
    if "Sample_ID" in df_temp.columns:
        df_temp.rename(columns={"Sample_ID": "Sample ID"}, inplace=True)
    if "Sample.Type" in df_temp.columns:
        df_temp.rename(columns={"Sample.Type": "Type"}, inplace=True)
    elif "Sample_Type" in df_temp.columns:
        df_temp.rename(columns={"Sample_Type": "Type"}, inplace=True)

# Combine all dataframes vertically
df = pd.concat(
    [df_23, df_20, df_22, df_24, df_25], ignore_index=True, sort=False
)

# Clean up Datetime formatting
df["Datetime"] = df["Date"].astype(str) + " " + df["Time"].astype(str)
df["Datetime"] = pd.to_datetime(df["Datetime"], format="mixed", errors="coerce")
df = df[df["Datetime"].notna()]

print(f"Interannual dataframe successfully compiled. Total samples: {df.shape[0]}")
print(f"Available columns: {list(df.columns)}")

# Save compiled RI18-25 grab sample chem file in Data\ directory as a csv
# file name = "RI18-25-joined.csv
output_file_path = output_dir / "RI18-25-joined.csv"

# Save the dataframe to CSV (index=False prevents saving row numbers)
df.to_csv(output_file_path, index=False)

Interannual dataframe successfully compiled. Total samples: 1145
Available columns: ['Sample ID', 'Site', 'Date', 'Time', 'Type', 'Type2', 'Index-notes', 'ICP-notes', 'Fe_mg_L', 'Mn_mg_L', 'Cu_mg_L', 'Zn_mg_L', 'Si_mg_L', 'K_mg_L', 'P_mg_L', 'Mg_mg_L', 'Na_mg_L', 'Al_mg_L', 'Ca_mg_L', 'F_mg_L', 'Cl_mg_L', 'NO2_mg_L', 'Br_mg_L', 'NO3_mg_L', 'PO4_mg_L', 'SO4_mg_L', 'IC-notes', 'NRS_LWIA_notes', 'dD', 'd18O', 'iso-notes', 'TOC run date', 'DOC_mg_L', 'TOC notes', 'Type3', 'S. No.', 'Transect', 'Depth', 'Pit', 'NO3_N_mg_L', 'SO4_S_mg_L', 'PO4_P_mg_L', 'Time Zone', 'Freezer bag #', 'UVM ICPOES prep date', 'ICPOES-notes-MED', 'Ba_mg_L', 'S_mg_L', 'Sr_mg_L', 'IC-notes_MED', 'Isotope/ICP pick list #', 'Acidified for ICPOES? (Y or N)', 'MED Index Notes', 'NRS Lab Comments', 'Datetime']
